In [1]:
# Instalar NLTK en Google Colab
!pip -q install nltk

import re
import nltk
import unicodedata
from nltk.tokenize import wordpunct_tokenize

print("NLTK instalado correctamente.")
print("Versión de Python compatible con el proyecto:", __import__("sys").version.split()[0])

NLTK instalado correctamente.
Versión de Python compatible con el proyecto: 3.13.16


In [2]:
# Información institucional
BASE_CONOCIMIENTO = {
    "nombre": "Clínica de Ojos Monterrey",
    "horarios": "Lunes a viernes de 8:00 a 17:00.",
    "direccion": "Av. Chepevera 777, Monterrey.",
    "servicios": [
        "Consulta oftalmológica general",
        "Revisión de la vista",
        "Seguimiento de salud visual",
        "Evaluación de molestias oculares"
    ],
    "requisitos": [
        "Identificación oficial, si corresponde.",
        "Estudios previos relacionados con la consulta, si los tienes.",
        "Lista de medicamentos que utilizas, si corresponde."
    ]
}

print("Base de conocimiento cargada.")

Base de conocimiento cargada.


In [3]:
def normalizar_texto(texto):
    """Convierte el texto a minúsculas y elimina acentos."""
    texto = texto.lower().strip()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        caracter for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )
    return texto


def obtener_tokens(texto):
    """Tokeniza el mensaje usando NLTK."""
    return wordpunct_tokenize(normalizar_texto(texto))


# Patrones sencillos para clasificar las intenciones.
PATRONES = {
    "urgencia": [
        r"\bperdi la vision\b",
        r"\bperdida repentina de vision\b",
        r"\bno veo de repente\b",
        r"\bdolor ocular intenso\b",
        r"\bquimico.*ojo\b",
        r"\bgolpe.*ojo\b",
        r"\bvision borrosa repentina\b",
        r"\bdestellos.*vision\b"
    ],
    "saludo": [
        r"\bhola\b",
        r"\bbuenos dias\b",
        r"\bbuenas tardes\b",
        r"\bbuenas noches\b",
        r"\bque tal\b"
    ],
    "despedida": [
        r"\badios\b",
        r"\bhasta luego\b",
        r"\bhasta pronto\b",
        r"\bnos vemos\b"
    ],
    "ayuda": [
        r"\bayuda\b",
        r"\bque puedes hacer\b",
        r"\bcomo funciona\b",
        r"\bopciones\b"
    ],
    "servicios": [
        r"\bservicios\b",
        r"\bque consultas ofrecen\b",
        r"\bque tipo de consultas\b",
        r"\brevision de la vista\b",
        r"\bexamen de la vista\b"
    ],
    "horarios": [
        r"\bhorarios?\b",
        r"\ba que hora abren\b",
        r"\ba que hora cierran\b",
        r"\bcuando abren\b",
        r"\bcuando cierran\b"
    ],
    "ubicacion": [
        r"\bdireccion\b",
        r"\bdonde estan\b",
        r"\bcomo llego\b",
        r"\bubicacion\b"
    ],
    "solicitar_cita": [
        r"\bquiero (?:agendar|solicitar|programar|sacar) una cita\b",
        r"\bnecesito una cita\b",
        r"\bagendar una consulta\b",
        r"\bquiero una consulta\b",
        r"\bsolicitar una cita\b"
    ],
    "requisitos": [
        r"\bque debo llevar\b",
        r"\bque necesito llevar\b",
        r"\brequisitos\b",
        r"\bdocumentos necesarios\b",
        r"\bque documentos\b"
    ],
    "modificar_cita": [
        r"\bcambiar mi cita\b",
        r"\bmodificar mi cita\b",
        r"\breprogramar\b",
        r"\bmover mi cita\b",
        r"\bcambiar el horario\b"
    ],
    "cancelar_cita": [
        r"\bcancelar mi cita\b",
        r"\bcancelar la consulta\b",
        r"\bquiero cancelar\b",
        r"\banular mi cita\b"
    ],
    "confirmacion": [
        r"\bsi\b",
        r"\bde acuerdo\b",
        r"\bcorrecto\b",
        r"\bconfirmo\b",
        r"\besta bien\b"
    ]
}


def clasificar_intencion(texto):
    texto_normalizado = normalizar_texto(texto)

    # Se priorizan las posibles urgencias oculares.
    for patron in PATRONES["urgencia"]:
        if re.search(patron, texto_normalizado):
            return "urgencia"

    # Se revisan las demás intenciones.
    for intencion, patrones in PATRONES.items():
        if intencion == "urgencia":
            continue

        for patron in patrones:
            if re.search(patron, texto_normalizado):
                return intencion

    return "desconocida"


# Comprobación rápida del clasificador
ejemplo = "Buenos días, quiero agendar una cita"
print("Tokens:", obtener_tokens(ejemplo))
print("Intención:", clasificar_intencion(ejemplo))

Tokens: ['buenos', 'dias', ',', 'quiero', 'agendar', 'una', 'cita']
Intención: saludo


In [4]:
def generar_respuesta(intencion, mensaje=""):
    if intencion == "saludo":
        return (
            f"Bienvenido a {BASE_CONOCIMIENTO['nombre']}. "
            "Puedo orientarte sobre citas, horarios, servicios, "
            "ubicación y requisitos. ¿Qué necesitas?"
        )

    elif intencion == "despedida":
        return "Gracias por comunicarte. Que tengas un buen día."

    elif intencion == "ayuda":
        return (
            "Puedo informarte sobre servicios, horarios, ubicación y "
            "requisitos, además de orientarte para solicitar, cambiar "
            "o cancelar una cita."
        )

    elif intencion == "servicios":
        servicios = "\n".join(
            f"- {servicio}"
            for servicio in BASE_CONOCIMIENTO["servicios"]
        )
        return "Estos son los servicios de ejemplo:\n" + servicios

    elif intencion == "horarios":
        return "Horario de atención de ejemplo: " + BASE_CONOCIMIENTO["horarios"]

    elif intencion == "ubicacion":
        return "Dirección de ejemplo: " + BASE_CONOCIMIENTO["direccion"]

    elif intencion == "solicitar_cita":
        return (
            "Para solicitar una cita normalmente se necesitan el nombre "
            "del paciente, el motivo de consulta y una fecha u horario "
            "preferido. Este prototipo no reserva horarios reales; "
            "debes confirmar la disponibilidad con el personal de la clínica."
        )

    elif intencion == "requisitos":
        requisitos = "\n".join(
            f"- {requisito}"
            for requisito in BASE_CONOCIMIENTO["requisitos"]
        )
        return (
            "Requisitos generales de ejemplo:\n"
            + requisitos
            + "\nConfirma los requisitos específicos con la clínica."
        )

    elif intencion == "modificar_cita":
        return (
            "Para cambiar una cita, contacta al personal administrativo "
            "e indica la fecha actual y el horario que prefieres. "
            "Este prototipo no modifica citas reales."
        )

    elif intencion == "cancelar_cita":
        return (
            "Para cancelar una cita, comunícate con el personal "
            "administrativo para identificar la reservación y confirmar "
            "la cancelación. Este prototipo no cancela citas reales."
        )

    elif intencion == "confirmacion":
        return (
            "Entendido. Para confirmar, cambiar o cancelar una cita real, "
            "es necesario verificar la solicitud con el personal de la clínica."
        )

    elif intencion == "urgencia":
        return (
            "Los síntomas oculares repentinos, como la pérdida súbita de "
            "visión, el dolor intenso o una lesión química, pueden requerir "
            "atención inmediata. Busca atención médica urgente o acude a "
            "un servicio de urgencias. No esperes una cita rutinaria ni "
            "dependas de este chatbot para evaluar la gravedad."
        )

    return (
        "No logré identificar tu solicitud. Puedes preguntar por horarios, "
        "servicios, ubicación, requisitos o gestión de citas. "
        "Si lo prefieres, contacta al personal de la clínica."
    )


def responder(mensaje):
    intencion = clasificar_intencion(mensaje)
    respuesta = generar_respuesta(intencion, mensaje)
    return intencion, respuesta


# Primera interacción
intencion, respuesta = responder("Hola")
print("Intención:", intencion)
print("Chatbot:", respuesta)

Intención: saludo
Chatbot: Bienvenido a Clínica de Ojos Monterrey. Puedo orientarte sobre citas, horarios, servicios, ubicación y requisitos. ¿Qué necesitas?


In [6]:
def iniciar_chat():
    print("Asistente de la clínica oftalmológica")
    print("Escribe 'salir' para terminar.\n")

    while True:
        mensaje = input("Usuario: ").strip()

        if normalizar_texto(mensaje) == "salir":
            print("Chatbot: Gracias por utilizar el asistente.")
            break

        if not mensaje:
            print("Chatbot: Escribe una consulta para continuar.")
            continue

        intencion, respuesta = responder(mensaje)
        print(f"Intención detectada: {intencion}")
        print(f"Chatbot: {respuesta}\n")


# Ejecuta esta función para conversar con el chatbot.
iniciar_chat()

Asistente de la clínica oftalmológica
Escribe 'salir' para terminar.

Usuario: Hola
Intención detectada: saludo
Chatbot: Bienvenido a Clínica de Ojos Monterrey. Puedo orientarte sobre citas, horarios, servicios, ubicación y requisitos. ¿Qué necesitas?

Usuario: Que horario manejan?
Intención detectada: horarios
Chatbot: Horario de atención de ejemplo: Lunes a viernes de 8:00 a 17:00.

Usuario: Hay citas disponibles para mañana?
Intención detectada: desconocida
Chatbot: No logré identificar tu solicitud. Puedes preguntar por horarios, servicios, ubicación, requisitos o gestión de citas. Si lo prefieres, contacta al personal de la clínica.

Usuario: donde se ubican?
Intención detectada: desconocida
Chatbot: No logré identificar tu solicitud. Puedes preguntar por horarios, servicios, ubicación, requisitos o gestión de citas. Si lo prefieres, contacta al personal de la clínica.

Usuario: Donde estan ubicados?
Intención detectada: ubicacion
Chatbot: Dirección de ejemplo: Av. Chepevera 777, 

In [7]:
# Cada caso contiene un mensaje y la intención que esperamos detectar.
CASOS_PRUEBA = [
    {
        "mensaje": "Hola, buenos días",
        "esperada": "saludo"
    },
    {
        "mensaje": "¿Qué servicios ofrecen?",
        "esperada": "servicios"
    },
    {
        "mensaje": "¿Cuáles son sus horarios?",
        "esperada": "horarios"
    },
    {
        "mensaje": "¿Dónde están ubicados?",
        "esperada": "ubicacion"
    },
    {
        "mensaje": "Quiero agendar una cita",
        "esperada": "solicitar_cita"
    },
    {
        "mensaje": "¿Qué debo llevar a la consulta?",
        "esperada": "requisitos"
    },
    {
        "mensaje": "Necesito cambiar mi cita",
        "esperada": "modificar_cita"
    },
    {
        "mensaje": "Quiero cancelar mi cita",
        "esperada": "cancelar_cita"
    },
    {
        "mensaje": "Sí, de acuerdo",
        "esperada": "confirmacion"
    },
    {
        "mensaje": "Perdí la visión de repente",
        "esperada": "urgencia"
    },
    {
        "mensaje": "¿Qué puedes hacer?",
        "esperada": "ayuda"
    },
    {
        "mensaje": "Hasta luego",
        "esperada": "despedida"
    },
    {
        "mensaje": "¿Cuál es el color del edificio?",
        "esperada": "desconocida"
    }
]

resultados = []

for numero, caso in enumerate(CASOS_PRUEBA, start=1):
    intencion, respuesta = responder(caso["mensaje"])
    correcto = intencion == caso["esperada"]

    resultados.append({
        "prueba": numero,
        "mensaje": caso["mensaje"],
        "esperada": caso["esperada"],
        "obtenida": intencion,
        "resultado": "CORRECTA" if correcto else "INCORRECTA",
        "respuesta": respuesta
    })

    print(f"Prueba {numero}: {caso['mensaje']}")
    print(f"Esperada: {caso['esperada']} | Obtenida: {intencion}")
    print(f"Resultado: {'CORRECTA' if correcto else 'INCORRECTA'}")
    print(f"Respuesta: {respuesta}")
    print("-" * 70)

total = len(resultados)
correctas = sum(r["resultado"] == "CORRECTA" for r in resultados)
incorrectas = total - correctas
tasa_exito = (correctas / total) * 100 if total else 0

print("\nRESUMEN DE PRUEBAS")
print(f"Total de pruebas: {total}")
print(f"Correctas: {correctas}")
print(f"Incorrectas: {incorrectas}")
print(f"Tasa de éxito: {tasa_exito:.2f}%")

Prueba 1: Hola, buenos días
Esperada: saludo | Obtenida: saludo
Resultado: CORRECTA
Respuesta: Bienvenido a Clínica de Ojos Monterrey. Puedo orientarte sobre citas, horarios, servicios, ubicación y requisitos. ¿Qué necesitas?
----------------------------------------------------------------------
Prueba 2: ¿Qué servicios ofrecen?
Esperada: servicios | Obtenida: servicios
Resultado: CORRECTA
Respuesta: Estos son los servicios de ejemplo:
- Consulta oftalmológica general
- Revisión de la vista
- Seguimiento de salud visual
- Evaluación de molestias oculares
----------------------------------------------------------------------
Prueba 3: ¿Cuáles son sus horarios?
Esperada: horarios | Obtenida: horarios
Resultado: CORRECTA
Respuesta: Horario de atención de ejemplo: Lunes a viernes de 8:00 a 17:00.
----------------------------------------------------------------------
Prueba 4: ¿Dónde están ubicados?
Esperada: ubicacion | Obtenida: ubicacion
Resultado: CORRECTA
Respuesta: Dirección de ejempl

In [8]:
# Casos adicionales para identificar posibles fallos del clasificador.
CASOS_ADICIONALES = [
    "BUENOS DÍAS",
    "Necesito reprogramar mi consulta",
    "Quiero saber la dirección",
    "Me cayó un producto químico en el ojo",
    "Hola, necesito cambiar mi cita",
    "Tengo una duda que no aparece en las opciones"
]

for mensaje in CASOS_ADICIONALES:
    intencion, respuesta = responder(mensaje)
    print(f"Mensaje: {mensaje}")
    print(f"Intención: {intencion}")
    print(f"Respuesta: {respuesta}\n")

Mensaje: BUENOS DÍAS
Intención: saludo
Respuesta: Bienvenido a Clínica de Ojos Monterrey. Puedo orientarte sobre citas, horarios, servicios, ubicación y requisitos. ¿Qué necesitas?

Mensaje: Necesito reprogramar mi consulta
Intención: modificar_cita
Respuesta: Para cambiar una cita, contacta al personal administrativo e indica la fecha actual y el horario que prefieres. Este prototipo no modifica citas reales.

Mensaje: Quiero saber la dirección
Intención: ubicacion
Respuesta: Dirección de ejemplo: Av. Chepevera 777, Monterrey.

Mensaje: Me cayó un producto químico en el ojo
Intención: urgencia
Respuesta: Los síntomas oculares repentinos, como la pérdida súbita de visión, el dolor intenso o una lesión química, pueden requerir atención inmediata. Busca atención médica urgente o acude a un servicio de urgencias. No esperes una cita rutinaria ni dependas de este chatbot para evaluar la gravedad.

Mensaje: Hola, necesito cambiar mi cita
Intención: saludo
Respuesta: Bienvenido a Clínica de 

In [9]:
# Guardar resultados.
import csv
from google.colab import files

nombre_archivo = "resultados_pruebas_chatbot.csv"

with open(nombre_archivo, "w", newline="", encoding="utf-8-sig") as archivo:
    campos = [
        "prueba", "mensaje", "esperada",
        "obtenida", "resultado", "respuesta"
    ]
    escritor = csv.DictWriter(archivo, fieldnames=campos)
    escritor.writeheader()
    escritor.writerows(resultados)

print(f"Archivo generado: {nombre_archivo}")
files.download(nombre_archivo)

Archivo generado: resultados_pruebas_chatbot.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>